In [29]:
from PIL import Image, ImageEnhance
import math
import os

def img_watermark(image_name, image_path):
    # 경로 설정
    parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(parent_dir, 'wm_uploads')
    os.makedirs(output_dir, exist_ok=True)

    # 1. 원본 이미지 처리
    original = Image.open(image_path)
    file_ext = os.path.splitext(image_name)[1].lower()
    
    # JPG 대응: RGB 모드로 변환
    if original.mode != 'RGBA':
        image = original.convert('RGBA')
    else:
        image = original.copy()

    # 2. 워터마크 로고 준비 (한 번만 로드)
    logo = Image.open("logo.png").convert("RGBA")
    alpha = logo.split()[3]
    alpha = ImageEnhance.Brightness(alpha).enhance(0.6)
    logo.putalpha(alpha)
    logo_width, logo_height = logo.size

    # 3. 워터마크 배치 계산
    width, height = image.size
    interval_x = math.trunc(width / 35)*10 if width > 600 else 200
    interval_y = 200 if height > 600 else math.trunc(height / 30)*10

    padding = 15
    usable_height = height - 2 * padding - logo_height
    num_lines = max(2, int(usable_height // interval_y) + 1)

    # 4. 워터마크 레이어 생성
    watermark_layer = Image.new("RGBA", (width, height), (0, 0, 0, 0))
    if num_lines == 2:
        y_coords = [padding, height - padding - logo_height]
    else:
        step = usable_height / (num_lines - 1)
        y_coords = [int(padding + i * step) for i in range(num_lines)]

    for y in y_coords:
        for x in range(0, width + interval_x, interval_x):
            watermark_layer.paste(logo, (x, y), logo)

    # 5. 회전 처리 (크기 유지)
    rotated_watermark = watermark_layer.rotate(
        45, 
        expand=False,  # 크기 변경 없음
        center=(width//2, height//2)
    )

    # 6. 이미지 합성
    watermarked = Image.alpha_composite(image, rotated_watermark)

    # 7. 저장 모드 결정
    save_path = os.path.join(output_dir, f"wm_{image_name}")
    
    if file_ext in ('.jpg', '.jpeg'):
        watermarked = watermarked.convert('RGB')  # 알파 채널 제거
        watermarked.save(save_path, quality=95, optimize=True)
    else:
        watermarked.save(save_path)

    return save_path


In [30]:
# !pip install PyMuPDF
import os
import fitz  # PyMuPDF 임포트
def pdf_watermark(pdf_name, path):
    
    # 상위 폴더 경로 계산
    parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(parent_dir, 'wm_uploads')
    
    # 출력 폴더 생성 (없을 경우)
    os.makedirs(output_dir, exist_ok=True)
    
    # 파일 경로 설정
    original_file = path
    watermark_file = 'WaterMark.pdf'
    new_file = os.path.join(output_dir, f"wm_{pdf_name}")
    
    # PDF 워터마킹 처리
    original_pdf = fitz.open(original_file)
    watermark_pdf = fitz.open(watermark_file)
    
    for page_num in range(len(original_pdf)):
        page = original_pdf[page_num]
        page.show_pdf_page(page.rect, watermark_pdf, 0)
    
    original_pdf.save(new_file)
    return new_file  # 전체 저장 경로 반환

In [31]:
# !pip install mysql-connector-python
# !pip install kiwipiepy

import mysql.connector
from collections import defaultdict
from datetime import datetime
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords
import re
import json
import os

# ───────── DB 설정─────────
DB_CONFIG = {
    "host":     "localhost",
    "user":     "root",
    "password": "1234",
    "database": "idealink",
    "charset":  "utf8mb4"
}

# ───────── 사용자 사전 경로 ─────────
USER_DICT_PATH = "user_dict.json"

# ───────── 의미 없는 불용어 사전 ─────────
STOPWORDS = {
    "활용", "개발", "발명", "사용", "제작", "연구", "분석", "대신", "기술", "이용", "시스템", "효과", "문제", "방법", "정보", "자료", 
    "프로젝트", "솔루션", "기반", "목표", "목적", "실험", "테스트", "검증", "구현", "설계", "제안", "확인", "수행", "평가", "수정", "개선",
    "환경", "서비스", "제도", "도입", "활동", "형태", "운영", "관리", "처리", "제공", "기획", "상황", "사례", "현황", "방안", "조사",
    "계획", "항목", "내용", "부분", "요소", "항목", "대책", "대응", "통한", "통해", "진행", "적용", "제시", "방식", "결과", "도출",
    "구조", "영향", "범위", "형성", "차이", "현상", "상태", "종류", "조건", "개념", "측면", "수준", "관계", "중심", "측정", "기초",
    "조정", "파악", "연계", "연결", "결정", "요약", "기준", "절차", "단계", "성과", "형성", "증가", "감소", "비교", "변화"
}

# ───────── DB 연결 헬퍼 ─────────
def connect_db():
    return mysql.connector.connect(**DB_CONFIG)

# ───────── 조회수 상위 40개 summary + views 가져오기 ─────────
def get_top_20_summary_views():
    query = "SELECT summary, view_count FROM post ORDER BY view_count DESC LIMIT 40"
    with connect_db() as conn, conn.cursor() as cur:
        cur.execute(query)
        return cur.fetchall()

# ───────── 사용자 사전 관리 ─────────
def load_user_dict():
    """JSON 파일에서 사용자 사전 로드"""
    if not os.path.exists(USER_DICT_PATH):
        return set()
    try:
        with open(USER_DICT_PATH, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    except:
        return set()

def save_user_dict(user_dict):
    """사용자 사전을 JSON 파일에 저장"""
    with open(USER_DICT_PATH, 'w', encoding='utf-8') as f:
        json.dump(list(user_dict), f, ensure_ascii=False)

# ───────── 합성어 탐지 및 등록 (띄어쓰기 기반) ─────────
def register_compounds_from_spaced_words(text, user_dict, new_compounds, kiwi):
    """
    띄어쓰기 기준으로 분할 후, 4글자 이상 단어에서 명사 조합을 합성어로 등록
    """
    # 띄어쓰기로 단어 분할
    words = text.split()
    
    for word in words:
        # 4글자 이상인 경우만 분석
        if len(word) < 4:
            continue
            
        # 형태소 분석
        tokens = kiwi.tokenize(word)
        compound_candidate = []
        
        # 연속된 명사(NNG/NNP) 탐지
        for token in tokens:
            if token.tag in ['NNG', 'NNP']:
                compound_candidate.append(token.form)
            else:
                if len(compound_candidate) >= 2:
                    # 명사 조합으로 합성어 생성
                    compound = ''.join(compound_candidate)
                    if compound not in user_dict and compound not in new_compounds:
                        new_compounds.add(compound)
                    break
                compound_candidate = []
        
        # 문장 끝 처리
        if len(compound_candidate) >= 2:
            compound = ''.join(compound_candidate)
            if compound not in user_dict and compound not in new_compounds:
                new_compounds.add(compound)



# ───────── 단어 정제 함수 ─────────
def clean_word(word):
    """이모지, 특수문자"""
    # 한글 완성형 또는 영어 알파벳만 추출
    valid = re.sub(r'[^가-힣a-zA-Z]', '', word)
    return valid if valid and len(valid) >= 2 else None

# ───────── 키워드 추출 함수 (수정) ─────────
def extract_keywords(summary_views, top_k=40):
    word_score = defaultdict(int)
    kiwi = Kiwi()
    stopwords = Stopwords()
    
    # 1. 사용자 사전 로드 및 Kiwi에 등록
    user_dict = load_user_dict()
    for word in user_dict:
        kiwi.add_user_word(word, "NNG")
    
    # 2. 이번 실행에서 발견된 새로운 합성어
    new_compounds = set()
    
    # 3. 합성어 등록 단계
    for summary, _ in summary_views:  # 조회수는 사용하지 않음
        if not summary:
            continue
        # 띄어쓰기 기준 합성어 분석 및 등록
        register_compounds_from_spaced_words(summary, user_dict, new_compounds, kiwi)
    
    # 4. 새로운 합성어 사전에 저장
    if new_compounds:
        updated_dict = user_dict | new_compounds
        save_user_dict(updated_dict)
        print(f"✅ 새로운 합성어 {len(new_compounds)}개 등록: {', '.join(list(new_compounds)[:3])}...")
        
        # Kiwi에 새 합성어 등록
        for word in new_compounds:
            kiwi.add_user_word(word, "NNG")
    
    # 5. 키워드 추출 단계
    for summary, views in summary_views:
        if not summary:
            continue
            
        # 형태소 분석
        tokens = kiwi.tokenize(summary)
        
        # 단어 추출 (명사, 고유명사, 형용사, 영어)
        words = [
            token.form
            for token in stopwords.filter(tokens)
            if token.form not in STOPWORDS  # 의미 없는 단어 필터링
            if token.tag in ['NNG', 'NNP', 'VA', 'SL']
        ]
        
        # 단어 정제 및 점수 누적
        for raw_word in words:
            cleaned_word = clean_word(raw_word)
            if cleaned_word:
                word_score[cleaned_word] += views
    
    # 6. 상위 키워드 추출
    keywords = sorted(word_score.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{"word": k[0], "score": k[1]} for k in keywords]

# ───────── 키워드 갱신 함수 ─────────
def refresh_keywords():
    global KEYWORDS_DATA
    rows = get_top_20_summary_views()
    KEYWORDS_DATA = extract_keywords(rows)
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] ✅ 키워드 데이터 갱신 완료")

In [ ]:
# !pip install flask
# !pip install flask-apscheduler
# !pip install flask-cors
from flask import Flask, request, jsonify
from flask import Response
from flask_cors import CORS
from concurrent.futures import ThreadPoolExecutor
from flask_apscheduler import APScheduler

app = Flask(__name__)
app.config['JSON_AS_ASCII'] = False
CORS(app)
KEYWORDS_DATA = []

# Config 클래스 정의
class Config:
    SCHEDULER_API_ENABLED = True # 스케줄러 API를 활성화함

app.config.from_object(Config()) # Flask 앱에 정의한 Config 적용
scheduler = APScheduler() # APScheduler 인스턴스 생성
scheduler.init_app(app) # 생성한 스케줄러를 Flask 앱에 등록(초기화)

# 1시간마다 키워드 자동 갱신
# @scheduler.task('interval', id='refresh_keywords', hours=1)
# 테스트용 1분마다 키워드 자동 갱신
@scheduler.task('interval', id='refresh_keywords', minutes=1)
def scheduled_refresh():
    refresh_keywords()

scheduler.start()

@app.route('/watermark', methods=['GET'])
def watermark():
    try:
        files = request.get_json()['files']
        print("========================================================")
        print("받은 files:", files)
        if not files or not isinstance(files, list):
            return jsonify({"error": "Invalid file list"}), 400

        file_info = [
            (file['filename'], file['path'], file['path'].split(".")[-1].lower())
            for file in files
        ]

        print("========================================================")
        print("정제한 files:", file_info)

        # 쓰레드를 통해 다중 처리
        processed_paths = []
        with ThreadPoolExecutor() as executor:
            futures = []
            for filename, path, ext in file_info:
                if ext == 'pdf':
                    futures.append(executor.submit(pdf_watermark, filename, path))
                else:
                    futures.append(executor.submit(img_watermark, filename, path))
            
            for future in futures:
                result = future.result()
                processed_paths.append(result)

        return jsonify({"wm_path": processed_paths}), 200

    except Exception as e:
        print("워터마크 오류 : ", e)
        return jsonify({"error": str(e)}), 500

# ───────── REST API ─────────
@app.route("/keywords")
def api_keywords():
    try:
        return jsonify(KEYWORDS_DATA)
    except Exception as e:
        print("키워드 오류 : ", e)
        return jsonify({"error": str(e)}), 500


# 서버 가동
if __name__ == "__main__":
    refresh_keywords() # 서버 시작시 키워드 갱신
    app.run()

[2025-06-25 09:57:44] ✅ 키워드 데이터 갱신 완료
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [25/Jun/2025 09:57:46] "GET /keywords HTTP/1.1" 200 -


[2025-06-25 09:58:01] ✅ 키워드 데이터 갱신 완료
[2025-06-25 09:58:17] ✅ 키워드 데이터 갱신 완료
[2025-06-25 09:58:25] ✅ 키워드 데이터 갱신 완료
[2025-06-25 09:58:28] ✅ 키워드 데이터 갱신 완료
[2025-06-25 09:58:31] ✅ 키워드 데이터 갱신 완료
[2025-06-25 09:58:35] ✅ 키워드 데이터 갱신 완료


127.0.0.1 - - [25/Jun/2025 09:58:38] "GET /keywords HTTP/1.1" 200 -


[2025-06-25 09:58:40] ✅ 키워드 데이터 갱신 완료
[2025-06-25 09:58:44] ✅ 키워드 데이터 갱신 완료


...

[2025-06-25 12:00:44] ✅ 키워드 데이터 갱신 완료


127.0.0.1 - - [25/Jun/2025 12:00:59] "GET /keywords HTTP/1.1" 200 -


[2025-06-25 12:01:00] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:01:17] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:01:25] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:01:28] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:01:31] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:01:35] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:01:39] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:01:44] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:02:00] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:02:17] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:02:25] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:02:28] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:02:31] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:02:35] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:02:39] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:02:44] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:03:00] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:03:17] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:03:25] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:03:28] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:03:31] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:03:35] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:03:39] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:03:44] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:04:00] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:04:17] ✅ 키워드 데이터 갱신 완료
[2025-06-25 

127.0.0.1 - - [25/Jun/2025 12:10:06] "GET /keywords HTTP/1.1" 200 -


[2025-06-25 12:10:17] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:10:25] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:10:28] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:10:31] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:10:35] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:10:39] ✅ 키워드 데이터 갱신 완료
[2025-06-25 12:10:44] ✅ 키워드 데이터 갱신 완료
